# Klasifikasi DemogPairs Menggunakan ViT (Umur) & Gaussian Naive Bayes

In [1]:
import numpy as np
import utils as u
import joblib
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from tqdm import tqdm

joblib.parallel_backend('threading')

## Load Dataset

In [2]:
data = u.load_demogpairs()
pd.DataFrame(data)

,db_code,image_path,full_path,label,label_idx
0,CWF,able_wanamakok/002.jpg,dataset/demogpairs/images\able_wanamakok/002.jpg,Asian_Females,5
1,CWF,able_wanamakok/004.jpg,dataset/demogpairs/images\able_wanamakok/004.jpg,Asian_Females,5
2,CWF,able_wanamakok/007.jpg,dataset/demogpairs/images\able_wanamakok/007.jpg,Asian_Females,5
3,CWF,able_wanamakok/008.jpg,dataset/demogpairs/images\able_wanamakok/008.jpg,Asian_Females,5
4,CWF,able_wanamakok/012.jpg,dataset/demogpairs/images\able_wanamakok/012.jpg,Asian_Females,5
...,...,...,...,...,...
10795,CWF,zachary_quinto/177.jpg,dataset/demogpairs/images\zachary_quinto/177.jpg,White_Males,3
10796,CWF,zachary_quinto/214.jpg,dataset/demogpairs/images\zachary_quinto/214.jpg,White_Males,3
10797,CWF,zachary_quinto/217.jpg,dataset/demogpairs/images\zachary_quinto/217.jpg,White_Males,3
10798,CWF,zachary_quinto/218.jpg,dataset/demogpairs/images\zachary_quinto/218.jpg,White_Males,3


## Load Fitur

In [3]:
features = joblib.load('features/demogpairs_vit-age.pkl')
print('Jumlah fitur per gambar:', np.array(features[list(features.keys())[0]]).shape[0])

Jumlah fitur per gambar: 768


## Split Data

In [4]:
X = np.array([features[d['image_path']] for d in data])
y = np.array([d['label_idx'] for d in data])
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
print((len(X_train), len(X_test)))

(8640, 2160)


## Kombinasi Parameter

In [5]:
var_smoothing_values = np.logspace(-9, 2, 40)  # dari 1e-9 sampai 1e2, 40 nilai

grid_params = [
    {
        'scaler': [None, MinMaxScaler()],
        'pca': [None, PCA(n_components=0.5), PCA(n_components=0.75)],
        
        'classifier': [GaussianNB()],
        'classifier__var_smoothing': var_smoothing_values
    },
]

pipeline = Pipeline(steps=[
    ('scaler', None),
    ('pca', None),
    ('classifier', None)
])

skv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy', 
    'f1': 'f1_macro', 
    'precision': 'precision_macro', 
    'recall': 'recall_macro',

}

grid_models = {}
for params in grid_params:
    key = str(params['classifier'][0]).split('(')[0]
    grid_models[key] = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        cv=skv, refit='accuracy',
        scoring=scoring, n_jobs=int(joblib.cpu_count() * 0.6),
        verbose=1, error_score='raise',
        return_train_score=True
    )
    print(f'{key}: {len(ParameterGrid(params))} kombinasi')

GaussianNB: 240 kombinasi


## Klasifikasi

In [6]:
evaluation_results, fold_results = u.evaluate_models(
    grid_models,
    X_train, y_train,
    X_test, y_test,
    target_names=u.demogpairs_classes,
    model_prefix='models/clf_demogpairs_gnb_vit-age_',
    results_path='results/demogpairs_gnb_vit-age_'
)

sorted_results = pd.DataFrame(evaluation_results).sort_values(by='test_accuracy', ascending=False).to_dict('records')
u.html_br()
_dtable = u.display_table(sorted_results)

Evaluating: GaussianNB


{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.000437547937507418), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}


Accuracy  : 0.6962962962962963
Precision : 0.697890477865101
Recall    : 0.6962962962962962
F1 Score  : 0.6951754820058508
               precision    recall  f1-score   support

Asian_Females     0.7099    0.7139    0.7119       360
  Asian_Males     0.6921    0.6056    0.6459       360
Black_Females     0.7418    0.6306    0.6817       360
  Black_Males     0.6512    0.7000    0.6747       360
White_Females     0.6941    0.7500    0.7210       360
  White_Males     0.6983    0.7778    0.7359       360

     accuracy                         0.6963      2160
    macro avg     0.6979    0.6963    0.6952      2160
 weighted avg     0.6979    0.6963    0.6952      2160



Class,OvR Accuracy,Precision,Recall,F1-Score,Support
Asian_Females,0.9037037037037037,0.7099447513812155,0.7138888888888889,0.7119113573407202,360
Asian_Males,0.8893518518518518,0.692063492063492,0.6055555555555555,0.6459259259259259,360
Black_Females,0.9018518518518519,0.7418300653594772,0.6305555555555555,0.6816816816816816,360
Black_Males,0.8875,0.6511627906976745,0.7,0.674698795180723,360
White_Females,0.9032407407407408,0.6940874035989717,0.75,0.7209612817089452,360
White_Males,0.9069444444444444,0.6982543640897756,0.7777777777777778,0.7358738501971092,360


Confusion matrix saved: images\cm_gnb_vit-age_GaussianNB.png



Confusion Matrix:
                         Asian_Females       Asian_Males     Black_Females       Black_Males     White_Females       White_Males
       Asian_Females               257                37                21                16                28                 1
         Asian_Males                46               218                 6                42                13                35
       Black_Females                23                11               227                38                52                 9
         Black_Males                 2                16                26               252                 7                57
       White_Females                29                 9                21                12               270                19
         White_Males                 5                24                 5                27                19               280


model_name,model_file_path,best_parameters,test_accuracy,test_f1,test_precision,test_recall,parameter_combinations
GaussianNB,models/clf_demogpairs_gnb_vit-age_GaussianNB.pkl,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.000437547937507418), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.6962962962962963,0.6951754820058508,0.697890477865101,0.6962962962962962,240


In [7]:
model, training_time = u.load_object('models/clf_demogpairs_gnb_vit-age_GaussianNB.pkl')
u.h(5, 'Waktu Pelatihan (Jobs)')
u.seconds_to_time(round(training_time))

{'input_seconds': 475.0,
 'days': 0,
 'hours': 0,
 'minutes': 7,
 'seconds': 55.0,
 'text': '0 hari 0 jam 7 menit 55.0 detik'}

In [8]:
u.h(5, 'Waktu Pelatihan')
times = [fr['Train Time Mean'] * 5 for fr in fold_results]
u.seconds_to_time(round(np.sum(times) + model.refit_time_))

{'input_seconds': 2410.0,
 'days': 0,
 'hours': 0,
 'minutes': 40,
 'seconds': 10.0,
 'text': '0 hari 0 jam 40 menit 10.0 detik'}

In [9]:
_dtable = u.display_table(fold_results, n_items=[4, 4], column_widths=['5%', '45%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%'])

No,Params,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Accuracy Mean,F1 Score Mean,Precision Mean,Recall Mean,Train Time Mean
1,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.000437547937507418), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.7002,0.6817,0.6875,0.6927,0.6973,0.6919,0.6902,0.694,0.6919,2.5738
2,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.00022854638641349884), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.6979,0.6823,0.6858,0.6927,0.6968,0.6911,0.6896,0.6929,0.6911,3.0914
3,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(3.257020655659783e-05), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.6991,0.6817,0.6834,0.6944,0.6927,0.6903,0.6889,0.692,0.6903,3.1135
4,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(1.7012542798525893e-05), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.6991,0.6817,0.6829,0.6944,0.6927,0.6902,0.6888,0.6919,0.6902,2.3507
...,...,...,...,...,...,...,...,...,...,...,...
237,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(52.233450742668325), 'pca': None, 'scaler': 'MinMaxScaler'}",0.39,0.3947,0.3837,0.3848,0.3964,0.3899,0.3496,0.4516,0.3899,0.3753
238,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(100.0), 'pca': None, 'scaler': 'MinMaxScaler'}",0.3906,0.3941,0.3848,0.3843,0.3953,0.3898,0.3492,0.452,0.3898,0.723
239,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(52.233450742668325), 'pca': None, 'scaler': None}",0.3912,0.3976,0.3773,0.3819,0.3976,0.3891,0.3502,0.4462,0.3891,0.5858
240,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(100.0), 'pca': None, 'scaler': None}",0.3912,0.397,0.3785,0.3814,0.397,0.389,0.3497,0.4458,0.389,0.8772
